# Generate an ExPRES Catalog for Cosmographia

This notebook reads an ExPRES CDF and flyby ephemerides, then creates Cosmographia catalogs and two video scripts (close-up and top-down views). Mission settings and SPICE kernels are defined in `cosmographia_missions_kernels.json`; reusable generation functions are in `cosmographia_utils.py`.

## Workflow

1. Enter the Cosmographia installation path, spacecraft, and flyby name.
2. If the flyby is not already configured, add it to the required JSON files as described below.
3. Provide the ExPRES CDF when prompted.
4. Choose the frequencies to export. `sample` selects four regularly spaced frequencies by default; `nearest` selects the available frequency closest to the requested value; `all` may produce a very large catalog.
5. Find the generated files under `cosmographia_movies/<flyby>/cosmo/`.

The outputs include the source catalog, spacecraft and SPICE configurations, a scenario file, and video-generation scripts.

## Mission Configuration

Supported spacecraft, Cosmographia frames, camera fields of view, and SPICE kernels are defined in `cosmographia_missions_kernels.json`. Add a mission profile there before using a new spacecraft. `spacecraft.json` records mission names recognized by the occultation tools.

## Cosmographia Documentation

[NAIF Cosmographia components and downloads](https://naif.jpl.nasa.gov/naif/cosmographia_components.html)

[CosmoGuide documentation](https://cosmoguide.org/)

[ESA Cosmographia seminar document (PDF)](https://www.cosmos.esa.int/documents/522118/522182/ESA-SCI-SPICE-HO-170309_Cosmographia_seminar-v1-20170309.pdf/a1acec29-dd69-a642-8a13-7da707b91b69)

In [1]:
import importlib
import json
import os
import sys
from pathlib import Path

import numpy
from astropy.units import Unit as u

current_dir = Path.cwd().resolve()
root_path = next(
    (path for path in (current_dir, *current_dir.parents)
     if (path / "Tutorial3_expres-cosmographia.ipynb").is_file()),
    None,
)
if root_path is None:
    raise FileNotFoundError(
        "Unable to find Tutorial3_expres-cosmographia.ipynb in the current directory "
        f"or its parents ({current_dir})."
    )
os.chdir(root_path)
if str(root_path) not in sys.path:
    sys.path.insert(0, str(root_path))

import occultation
from cosmographia_utils import (
    build_expres_catalog,
    build_movie_script,
    build_scenario_dict,
    build_spacecraft_catalog,
    build_spice_dict,
    load_mission_config,
    select_frequency_indices,
    write_json,
)

/Users/clouis/anaconda3/envs/expres-occultation/lib/python3.9/site-packages/spacepy/time.py:2448: UserWarning: Leapseconds may be out of date. Use spacepy.toolbox.update(leapsecs=True)
  _read_leaps()


In [2]:
R_jup = (1 * u("R_jup")).to("km").value
R_sat = 58232.0

 ### Defining local paths

In [3]:
cosmographia_path = input("Please enter the path to your Cosmographia installation: ") or "/Applications/Cosmographia" 

In [4]:
spacecraft_input = input("Please enter spacecraft name (for JUICE, enter JUICE): ").strip()
mission_config = load_mission_config(root_path / "cosmographia_missions_kernels.json", spacecraft_input)
Spacecraft_name = mission_config["canonical_name"]
flyby_name = input("Please enter flyby name: ").strip()

## Mission configuration

`spacecraft.json` records mission names recognized by the occultation tools. New mission will automatically be added to this file.

The supported missions, Cosmographia frames, camera fields of view, and SPICE kernels are defined in `cosmographia_missions_kernels.json`. Add a mission there if the code asks you to do so in the next code block.



### Download the Required SPICE Kernels

Download the mission-specific SPICE kernels listed in `cosmographia_missions_kernels.json` from the [NAIF SPICE archive](https://naif.jpl.nasa.gov/pub/naif/). Place each kernel in your Cosmographia installation (`cosmographia_path` directory) under subdirectories structure expected by the selected mission profile (see `cosmographia_missions_kernels.json`).

Cassini uses a separate Cosmographia SPICE catalog JSON for each year. The notebook selects the catalog matching the year of the ExPRES CDF; make sure that the corresponding `spice_Cassini_<year>.json` file and all kernels it references are available in the configured installation. These files are constructed based on the `cas_<year>_v<version>.tm` files.

Any changes made to the SPICE kernel must be reflected in the `cosmographia_missions_kernels.json` file.

In [5]:
with (root_path / "spacecraft.json").open(encoding="utf-8") as f:
    spacecraft_json = json.load(f)

if Spacecraft_name not in spacecraft_json["available_missions"]:
    print("Mission not in spacecraft.json, adding it now... You will also need to add the mission to cosmographia_missions_kernels.json.")
    spacecraft_json["available_missions"].append(Spacecraft_name)
    write_json(root_path / "spacecraft.json", spacecraft_json)

## Add a Flyby to `flybys_list.json` (if missing)

If the selected flyby is not listed, add an entry keyed by its flyby name. Each entry must define:

- `spacecraft`: the mission name exactly as listed in `spacecraft.json` (for example, `JUICE`, `Juno`, `GLL`, or `Cassini`).
- `moon`: the target moon name used by the ephemeris configuration (for example, `Europa` or `Ganymede`).
- `start_time` and `end_time`: the flyby time interval in ISO format `YYYY-MM-DDTHH:mm:ss`.

The next cell can create this entry interactively. Check the names and UTC times before continuing; `occultation.py` reads these values when the flyby object is created.

In [6]:
with open("flybys_list.json") as f:
    flyblys_list_json = json.load(f)

if flyby_name not in flyblys_list_json.keys():
    moon_input = input("Please enter the moon involved in the flyby: ")
    start_time_input = input("Please enter the START time of the flyby (format: YYYY-MM-DDTHH:mm:ss): ")
    end_time_input = input("Please enter the END time of the flyby (format: YYYY-MM-DDTHH:mm:ss): ")
    flyblys_list_json[f'{flyby_name}'] = {
        'spacecraft': f'{Spacecraft_name}',
        'moon': f'{moon_input}',
        'start_time': f'{start_time_input}',
        'end_time': f'{end_time_input}'
    }
    with open("flybys_list.json", "w") as f:
        json.dump(flyblys_list_json, f)





## Adding Ephemerides to `ephem-wgc.json` (if missing)

If the flyby is not listed, add an entry keyed by its flyby name. It must map the spacecraft name and moon name from `flybys_list.json` to their corresponding WebGeocalc state-vector CSV files. Store both CSV files in the project's `ephem/` directory; when prompted, enter each filename relative to that directory, without the `ephem/` prefix.

For example, a JUICE flyby of Europa should contain keys `JUICE` and `Europa`, with values such as `ephem/juice_state_vector.csv` and `ephem/europa_state_vector.csv`. The names must match the spacecraft and moon fields in `flybys_list.json` exactly. The next cell creates a basic entry if the flyby is absent; verify the generated JSON paths before continuing.

In [7]:
with open("ephem-wgc.json") as f:
    ephem_wgc_json = json.load(f)

if flyby_name not in ephem_wgc_json.keys():
    moon_input = input("Please enter the moon involved in the flyby: ")
    ephem_moon_input = input("Please enter the Moon ephemeris file corresponding to the flyby (the one you used for the occultation pipeline)")
    ephem_spacecraft_input = input("Please enter the Spacecraft ephemeris file corresponding to the flyby (the one you used in ExPRES and for the occultation pipeline)")
    print("PLEASE STORE BOTH OF YOUR EPHEMERIS FILE IN THE ephem directory")
    ephem_wgc_json[f'{flyby_name}'] = {
            f'{Spacecraft_name}': f'ephem/{ephem_spacecraft_input}',
            f'{moon_input}': f'ephem/{ephem_moon_input}',
        }
    with open("ephem-wgc.json", "w") as f:
        json.dump(ephem_wgc_json, f)

# flyby_table = get_flyby_table()

## Loading and Reading CDF file

The code will ask you to enter the full path and name of the ExPRES CDF containing source positions

In [8]:
expres_cdf = {}
cdf_file_path = input(
    "Please enter the full path and name of the ExPRES CDF containing source positions: "
).strip()
if not Path(cdf_file_path).is_file():
    raise FileNotFoundError(f"ExPRES CDF not found: {cdf_file_path}")

# Reload occultation so flybys and ephemerides added above are visible.
occultation = importlib.reload(occultation)
flyby_object = occultation.Flyby(
    flyby_name,
    "wgc",
    "wgc",
    expres_file=cdf_file_path,
)

/Users/clouis/anaconda3/envs/expres-occultation/lib/python3.9/site-packages/numpy/linalg/linalg.py:2582: RuntimeWarning: overflow encountered in multiply
  s = (x.conj() * x).real


(241, 42, 722)
0.009374531031508452
0.18535532699753918


In [9]:
print(f'Shape of source_xyz = {flyby_object.source.xyz[:].shape}')
print(f'Number of time steps = {flyby_object.source.Time[:].shape}')
dt = flyby_object.source.Time[1] - flyby_object.source.Time[0]
print(f'Time step = {dt}')

Shape of source_xyz = (241, 42, 722, 3)
Number of time steps = (241,)
Time step = 0:01:00


In [10]:
Freq = numpy.asarray(flyby_object.source.Freq[...])
frequency_mode = input(
    "Frequency selection: [a]ll, [n]earest to a requested frequency, "
    "or [s]ample regularly (default: n): "
).strip().lower() or "n"

frequency_count = 4
default_frequency = 0.6 if Spacecraft_name.lower() == "cassini" else 9.0
requested_frequency = None
if frequency_mode in ("n", "nearest"):
    requested_frequency = float(
        input(f"Requested frequency in MHz (default: {default_frequency:g}): ")
        or default_frequency
    )
elif frequency_mode in ("s", "sample", "sampled"):
    frequency_count = int(input("Number of regularly spaced frequencies (default: 4): ") or "4")

freq_indices = select_frequency_indices(
    Freq,
    frequency_mode,
    requested_frequency=requested_frequency,
    count=frequency_count,
)
if freq_indices is None:
    print(f"All {len(Freq)} frequencies selected. The output catalog may be large.")
else:
    print("Selected frequencies (MHz): " + ", ".join(f"{Freq[index]:g}" for index in freq_indices))

dt = flyby_object.source.Time[1] - flyby_object.source.Time[0]

R_frame = R_jup if mission_config["center"] == "Jupiter" else R_sat
cosmo = build_expres_catalog(
    flyby_name=flyby_name,
    times=flyby_object.source.Time[...],
    source_positions=flyby_object.source.xyz[...],
    dt=dt,
    mission_config=mission_config,
    distance_scale=R_frame,
    frequency_indices=freq_indices,
)
print(f"Generated {len(cosmo['items'])} Cosmographia items.")

Selected frequencies (MHz): 0.1008, 0.398481, 1.42796, 5.645
Generated 4445 Cosmographia items.


In [11]:
output_path_catalog = root_path / "cosmographia_movies" / flyby_name / "cosmo" / "catalog"
cosmo_expres_catalog_filename = f"cosmo_expres_{flyby_name}.json"
cosmo_expres_catalog_path = write_json(
    output_path_catalog / cosmo_expres_catalog_filename,
    cosmo,
)
print(f"ExPRES catalog written to: {cosmo_expres_catalog_path}")

ExPRES catalog written to: /Users/clouis/Documents/Routine/ExPRES_JUICE/expres-occultation/cosmographia_movies/C09/cosmo/catalog/cosmo_expres_C09.json


In [12]:
Spacecraft_name_for_file = mission_config["canonical_name"]
center_frame = mission_config["center"]
spacecraft_dict = build_spacecraft_catalog(
    mission_config,
    flyby_name,
    flyby_object.source.Time[0],
    flyby_object.source.Time[-1],
)

output_path_config = root_path / "cosmographia_movies" / flyby_name / "cosmo" / "config"
spacecraft_config_filename = f"spacecraft_{Spacecraft_name_for_file}_{flyby_name}.json"
spacecraft_config_path = write_json(
    output_path_config / spacecraft_config_filename,
    spacecraft_dict,
)
print(f"Spacecraft config written to: {spacecraft_config_path}")

Spacecraft config written to: /Users/clouis/Documents/Routine/ExPRES_JUICE/expres-occultation/cosmographia_movies/C09/cosmo/config/spacecraft_GLL_C09.json


In [13]:
catalog_year = flyby_object.source.Time[-1].year
spice_dict = build_spice_dict(
    mission_config,
    cosmographia_path,
    year=catalog_year,
)

spice_kernel_for_cosmo_file = f"spice_{Spacecraft_name_for_file}_{flyby_name}.json"
spice_config_path = write_json(
    output_path_config / spice_kernel_for_cosmo_file,
    spice_dict,
)
print(f"SPICE config written to: {spice_config_path}")

SPICE config written to: /Users/clouis/Documents/Routine/ExPRES_JUICE/expres-occultation/cosmographia_movies/C09/cosmo/config/spice_GLL_C09.json


In [14]:
load_scenario_file = f"load_{Spacecraft_name_for_file}_{flyby_name}_001.json"
output_path_scenario = root_path / "cosmographia_movies" / flyby_name / "cosmo" / "scenarios"
scenario_path = output_path_scenario / load_scenario_file

scenarios_dict = build_scenario_dict(
    Spacecraft_name_for_file,
    flyby_name,
    [spice_config_path, spacecraft_config_path],
)
write_json(scenario_path, scenarios_dict)
print(f"Scenario written to: {scenario_path}")

Scenario written to: /Users/clouis/Documents/Routine/ExPRES_JUICE/expres-occultation/cosmographia_movies/C09/cosmo/scenarios/load_GLL_C09_001.json


In [15]:
output_path_movie = root_path / "cosmographia_movies" / flyby_name / "cosmo" / "movies"
output_path_movie.mkdir(parents=True, exist_ok=True)
fieldOfView_pov = mission_config["fov"]["pov"]
fieldOfView_top = mission_config["fov"]["top"]

In [16]:
SimulationDuration = (flyby_object.source.Time[-1]-flyby_object.source.Time[0]).total_seconds()

In [17]:
movie_file = f"cosmo_expres_{Spacecraft_name_for_file}_{flyby_name}_pov.m4v"
movie_path = output_path_movie / movie_file
script = build_movie_script(
    view="pov",
    spacecraft_name=Spacecraft_name_for_file,
    center=center_frame,
    start_time=flyby_object.source.Time[0],
    config_path=spacecraft_config_path,
    scenario_path=scenario_path,
    expres_catalog_path=cosmo_expres_catalog_path,
    movie_path=movie_path,
    fov=fieldOfView_pov,
    simulation_duration=SimulationDuration,
)

output_path_script = root_path / "cosmographia_movies" / flyby_name / "cosmo" / "scripts"
script_path = output_path_script / f"{Spacecraft_name_for_file}_{flyby_name}_pov.py"
output_path_script.mkdir(parents=True, exist_ok=True)
script_path.write_text(script, encoding="utf-8")
print(f"POV script written to: {script_path}")

POV script written to: /Users/clouis/Documents/Routine/ExPRES_JUICE/expres-occultation/cosmographia_movies/C09/cosmo/scripts/GLL_C09_pov.py


In [18]:
movie_file = f"cosmo_expres_{Spacecraft_name_for_file}_{flyby_name}_top.m4v"
movie_path = output_path_movie / movie_file
script = build_movie_script(
    view="top",
    spacecraft_name=Spacecraft_name_for_file,
    center=center_frame,
    start_time=flyby_object.source.Time[0],
    config_path=spacecraft_config_path,
    scenario_path=scenario_path,
    expres_catalog_path=cosmo_expres_catalog_path,
    movie_path=movie_path,
    fov=fieldOfView_top,
    simulation_duration=SimulationDuration,
)

output_path_script = root_path / "cosmographia_movies" / flyby_name / "cosmo" / "scripts"
script_path = output_path_script / f"{Spacecraft_name_for_file}_{flyby_name}_top.py"
output_path_script.mkdir(parents=True, exist_ok=True)
script_path.write_text(script, encoding="utf-8")
print(f"Top-view script written to: {script_path}")

Top-view script written to: /Users/clouis/Documents/Routine/ExPRES_JUICE/expres-occultation/cosmographia_movies/C09/cosmo/scripts/GLL_C09_top.py


## Run a Movie Script in Cosmographia

1. Start Cosmographia.
2. From the menu, select **File > Run Script...**.
3. Browse to `cosmographia_movies/<flyby>/cosmo/scripts/` and select the generated script ending in `_pov.py` for the close-up view or `_top.py` for the top-down view.

The script loads the generated catalogs, sets the view and simulation time, and records the movie in `cosmographia_movies/<flyby>/cosmo/movies/`.